<a href="https://colab.research.google.com/github/lostysky/Data-Engineering-codeboosters-internship-2026/blob/main/Day_07_Embeddings_SemanticSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install require libraries

In [ ]:
#Install the two new libaries needed for today
#chromadb: vector database (like sqlite but for ai embedding)
#sentence-transformers: converts text to 384-dimenstional number vectors
!pip install chromadb sentence-transformers -q
#-q means quite mode - reduces the amount of output printed
#You will see a progress bar. Wait until it says 'Successfully installed

print("Installation Completed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb
print("All libraries imported successfully!")
print(f"Chromadb version: {chromadb.__version__}")

DEMONSTRATION: Keyword Search vs Semantic Search

In [ ]:
#Imagine we have a small collection of documents about data
documents = [
    "ETL is used to clean and tranform data",  #doc0
    "A vehicle is a mode of transportation",   #doc1
    "Cars and trucks are popular automobiles", #doc2
    "SQL is used to query databases",          #doc3
    "Machine learning trains models on data",  #doc4
]

#Keyword Search
#Keyword Search: Check if the exact query word appears in the document

query_keyword = "vehicle"
print("=" * 60)
print(f"Keyword Search for: '{query_keyword}'")
print("=" * 60)

for i, doc in enumerate(documents):
  #lower() makes comparison case-insenstive
  #in checks if the word appears anywhere in he string
  if query_keyword.lower() in doc.lower():
    print(f"Found in document {i}: {doc}")
  else:
    print(f" Missed [doc_{i}]: {doc}")

print()
print("PROBLEM: doc2 talks about 'Cars and Trucks' - which are vehicles")
print("But the keyword missed it because it searched for the exact word vehicle")

Keyword Search for: 'vehicle'
 Missed [doc_0]: ETL is used to clean and tranform data
Found in document 1: A vehicle is a mode of transportation
 Missed [doc_2]: Cars and trucks are popular automobiles
 Missed [doc_3]: SQL is used to query databases
 Missed [doc_4]: Machine learning trains models on data

PROBLEM: doc2 talks about 'Cars and Trucks' - which are vehicles
But the keyword missed it because it searched for the exact word vehicle


In [ ]:
#More examples of keyword search failures
#These are all cases where meaning matches but do not

failure_examples = [
    {"query": "I feel sick",                   "misses": "I am unwell, patient has fever"},
    {"query": "How to cook rice",              "misses": "Steps to prepare rice"},
    {"query": "Vehicle",                       "misses": "car acceleration, automabile velocity"},
    {"query": "ML model accuracy",             "misses": "classification performance, prediction quality"},
]


print("KEYWORD SEARCH FAILURE CASES")
print("=" * 60)
for ex in failure_examples:
  #f-string: embeds variables inside the string using{}
  print(f"Query: '{ex['query']}'")
  print(f"  Misses: '{ex['misses']}'")
  print("-" * 40)

print()
print("SOLUTION: We need search that understans MEANING, not just characters.")
print("That is what EMBEDDINGS do.")

KEYWORD SEARCH FAILURE CASES
Query: 'I feel sick'
  Misses: 'I am unwell, patient has fever'
----------------------------------------
Query: 'How to cook rice'
  Misses: 'Steps to prepare rice'
----------------------------------------
Query: 'Vehicle'
  Misses: 'car acceleration, automabile velocity'
----------------------------------------
Query: 'ML model accuracy'
  Misses: 'classification performance, prediction quality'
----------------------------------------

SOLUTION: We need search that understans MEANING, not just characters.
That is what EMBEDDINGS do.


# *Key Vocabulary*

**vector** - *A list of numbers (our embedding are list of 384 numbers)*

**Dimension** - *One number in the list(384 dimensions = 384 numbers)*

**Cosine Similarity** - *A score from 0.0 to 1.0 - how similar two vector are*

**Embedding Model** - *The neural network that converts texts to vectors*

In [ ]:
#- 'v2' means : version 2
# Output: 384-dimenstional vectors
#note: first run downloads the model(`80MB). This is a on-time download.
print("Loading embedding model...  (may take 1-2 minutes on first run)")

model = SentenceTransformer('all-MiniLM-L6-v2')
#the model is now loaded and ready to convert text to embeddings
print("Model Loaded Successfully")
print(f'Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions')
#
#

Loading embedding model...  (may take 1-2 minutes on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully
Model produces vectors of size: 384 dimensions


/tmp/ipykernel_6835/1202065171.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions')


In [ ]:
#Define a single sentence to embed

sentence = "ETL is used to clean and tansform data"

#model.encode() converts text to a vector(list of numbers)
#Input: a string OR a list of strings
#Output: a Numpy array

embedding = model.encode(sentence)
#Let us explore the output
print(f"input Sentence: '{sentence}'")
print()
print(f"Embeeding type: {type(embedding)}")
#type shows this is a numpy.ndarray - an array of numbers

print(f"Embedding shape: {embedding.shape}")
#Shape (384,) means: 1 sentence * 384 numbers
print(f"First 10 numbers: {embedding[:10].round(4)}")
#These are the first 10 of 384 numbers
#.round(4) rounds to 4 decimal places for readability
print(f"Min value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")

input Sentence: 'ETL is used to clean and tansform data'

Embeeding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
First 10 numbers: [-0.0584  0.0355  0.0165 -0.029   0.0394 -0.0859  0.0379 -0.0053  0.0565
  0.0284]
Min value: -0.1335
Max value: 0.1731


In [ ]:
sentences = [
    "ETL is used to clean and transform  data",    #sentence0
    "Data transformation is a key pipleline step", #sentence1
    "The sky is blue and clouds are white",        #sentence2
]

#Embedd all three sentences at once
embeddings = model.encode(sentences)
#embeddings is a 2D NumPy array

print(f"Number of sentences: {len(sentences)}")
print(f"Shape of embedding array: {embeddings.shape}")
#Expected output: (3, 384), 3 = number of sentences, 384 - dimensions per sentences

print()
print("Each row is one sentence's embedding:")
for i, sent in enumerate(sentences):
  #embedd[i] gives i (sentence i's vector)
  print(f" Sentence {i}: Shape={embeddings[i].shape}, first 5 values={embeddings[i][:5].round(3)}")


Number of sentences: 3
Shape of embedding array: (3, 384)

Each row is one sentence's embedding:
 Sentence 0: Shape=(384,), first 5 values=[-0.078  0.054  0.022 -0.039  0.022]
 Sentence 1: Shape=(384,), first 5 values=[-0.033  0.051  0.013 -0.034 -0.071]
 Sentence 2: Shape=(384,), first 5 values=[0.054 0.06  0.075 0.049 0.05 ]


In [ ]:
def cosine_similarity(vec_a, vec_b):
  """Calculate cosine similarity between two vectors."""
  #np.dot: dot product of two arrays (multiply element-wise, then sum)
  dot_product = np.dot(vec_a, vec_b)

  #np.linalg norm: length(magnitude) of the vector
  norm_a = np.linalg.norm(vec_a)
  norm_b = np.linalg.norm(vec_b)

  #Divide dot product by product of lengths
  return dot_product / (norm_a * norm_b)

In [ ]:
#calculate the similarities
sim_01=cosine_similarity(embeddings[0],embeddings[1])
sim_02=cosine_similarity(embeddings[0],embeddings[2])
sim_12=cosine_similarity(embeddings[1],embeddings[2])
print("Cosine similarities")
print()
print(f"Sentence 0 :{sentences[0]}")
print(f"Sentence 1 :{sentences[1]}")
print(f"Sentence 2 :{sentences[2]}")
print()
print(f"Similarity (0 vs 1): {sim_01 :.4f} <- Expected: HIGH (same topic)")
print(f"Similarity (0 vs 2): {sim_02 :.4f} <- Expected: LOW (different topic)")
print(f"Similarity (1 vs 2): {sim_12 :.4f} <- Expected: LOW (different topic)")
print()
print("insights : sentence 0 and 1 have the differsnt words but similar meaning")
print("their cosine similarity score is high - the embedding captured the meaning!")

Cosine similarities

Sentence 0 :ETL is used to clean and transform  data
Sentence 1 :Data transformation is a key pipleline step
Sentence 2 :The sky is blue and clouds are white

Similarity (0 vs 1): 0.4026 <- Expected: HIGH (same topic)
Similarity (0 vs 2): -0.0226 <- Expected: LOW (different topic)
Similarity (1 vs 2): 0.0609 <- Expected: LOW (different topic)

insights : sentence 0 and 1 have the differsnt words but similar meaning
their cosine similarity score is high - the embedding captured the meaning!


In [ ]:
#my semtence
your_sentences= [
    "Today I ate biriyani",
    "Machine learning is the important subject",
    "vegetables is good for health"
]
your_embeddings=model.encode(your_sentences)

#check similarity of my sentence
ysim_01=cosine_similarity(your_embeddings[0],your_embeddings[1])
ysim_02=cosine_similarity(your_embeddings[0],your_embeddings[2])
ysim_12=cosine_similarity(your_embeddings[1],your_embeddings[2])
print("Cosine similarities")
print()
print(f"Sentence 0 :{your_sentences[0]}")
print(f"Sentence 1 :{your_sentences[1]}")
print(f"Sentence 2 :{your_sentences[2]}")
print()
print(f"Similarity (0 vs 1): {ysim_01 :.4f} <- Expected: LOW (different topic)")
print(f"Similarity (0 vs 2): {ysim_02 :.4f} <- Expected: HIGH (same topic)")
print(f"Similarity (1 vs 2): {ysim_12 :.4f} <- Expected: LOW (different topic)")
print()

#Interpretation
if ysim_01 > ysim_02 and ysim_01 > ysim_12:
  print("Sentence 0 is more similar to sentence 1")
  print(f"Similarity score: {ysim_01}")
else :
  print("Sentence 0 is not more similar to sentence 1")
  print(f"Similarity score: {ysim_01}")

Cosine similarities

Sentence 0 :Today I ate biriyani
Sentence 1 :Machine learning is the important subject
Sentence 2 :vegetables is good for health

Similarity (0 vs 1): 0.0366 <- Expected: LOW (different topic)
Similarity (0 vs 2): 0.2790 <- Expected: HIGH (same topic)
Similarity (1 vs 2): 0.0745 <- Expected: LOW (different topic)

Sentence 0 is not more similar to sentence 1
Similarity score: 0.0366477444767952


ChromaDB key:

Collection : Like a table in the sql - a named group of documents

Document : The text of your note/record

ID : A unique identifier for each document (like primary key)

MetaData : Extra information about the document ( eg: subject, author, data )

Distance : How far apart two vectors are - lower =more similar

Critical rule  : Distance vs Similarity.

chromadb ->returns the distance - not similarity

1. Distance 0.0 = perfect match (identical vectors)

2. Distance 1.0 = no match (completely different)

3. This is the opposite of cosine similarity score direction!

In [ ]:
#when you restart the kernel , data will diappear and must re add it
#For permanent storage : use chromadb.PersistentClient(path='./chroma_db')
chroma_client = chromadb.Client()

#get_or_create_collection : creates the collection or open the exist
#demo_notes is the name of the collection
collection=chroma_client.get_or_create_collection("demo_notes")

print("Chroma client created (in memory mode)")
print(f"collection name : demo_notes")

#count() returns how many documents in the collection
print(f"Documents in the collection : {collection.count()}")

Chroma client created (in memory mode)
collection name : demo_notes
Documents in the collection : 0


In [ ]:
#Let us add 5 sample documents
sample_docs=[
    "ETL stands for extract transform load - the core data engineering process",
    "SQL SELECT statements retrieve data from database tables",
    "machine learning models learn patterns from training data",
    "Python pandas library is used for data manipulation and cleaning",
    "Neural networks are inspired by how the human brain works",
]

sample_ids=["doc001","doc002","doc003", "doc004", "doc005"]

sample_metadata=[
    {"subject":"Data Engineering","topic":"ETL"},
    {"subject":"Data Engineering","topic":"SQL"},
    {"subject":"Machine Learning","topic":"ML basics"},
    {"subject":"Python","topic":"Pandas"},
    {"subject":"machine Learning","topic":"Neural networks"},
]

collection.add(
    documents=sample_docs, #the actual text content
    ids=sample_ids, #unique string
    metadatas=sample_metadata #optional metadata
)

print(f"Document added to collection!")
print(f"total documents now in the collection : {collection.count()}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.5MiB/s]


Document added to collection!
total documents now in the collection : 5


In [ ]:
query = "How do i clean and prepare the data?"
#Notice : this query does not contain the words : 'ETL' or 'PANDAS'
#But it means the same thing - lets see if semantic search finds
results=collection.query(
    query_texts=[query],
    n_results=3
)
print("RESULTS KEYS AVAILABLE")
print(list(results.keys()))

RESULTS KEYS AVAILABLE
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [ ]:
#display the result in the readable format
print(f"Query:'{query}")
print()

#results['documents'] is the lists of lists
#the outer list has one element per query (we had 1 query)
#the inner list has one element per result
#So the results['document'][0] is the list of matched documents for out 1st query
matched_docs = results['documents'][0] #List of matched texts
matched_ids=results['ids'][0] #List of matched IDs
matched_distances=results['distances'][0] #List of distance scores
matched_metadata=results['metadatas'][0] #List of metadata dicts

#zip() pairs elements from multiple lists together
#This is the same as going matched_docs[0], matched_ids[0], etc in a loop
for rank, (doc,doc_id, dist, meta) in enumerate(zip(matched_docs,matched_ids, matched_distances, matched_metadata)):
  print(f"rank {rank} | ID : {doc_id} | Distance : {dist:.4f}")
  print(f"subject: {meta['subject']} | Topic : {meta['topic']}")
  print(f"Document : {doc}")
  print()
print("Notice : the results are about ETL and pandas - exactly what 'clean and prepare data' means!")
print("Semantic searches found them even though the words are different")

Query:'How do i clean and prepare the data?

rank 0 | ID : doc004 | Distance : 1.1048
subject: Python | Topic : Pandas
Document : Python pandas library is used for data manipulation and cleaning

rank 1 | ID : doc002 | Distance : 1.5847
subject: Data Engineering | Topic : SQL
Document : SQL SELECT statements retrieve data from database tables

rank 2 | ID : doc003 | Distance : 1.6336
subject: Machine Learning | Topic : ML basics
Document : machine learning models learn patterns from training data

Notice : the results are about ETL and pandas - exactly what 'clean and prepare data' means!
Semantic searches found them even though the words are different


In [ ]:
filtered_query = "How do computers learn from examples?"

filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject": "Machine Learning"}
    #Where: a dictionary that filters
)

print(f"FILTERED QUERY: '{filtered_query}'")
print("Filter : Only Machine Learning documents")
print("="*60)

for rank, (doc, dist, meta) in enumerate(zip(filtered_results["documents"][0], filtered_results["distances"][0], filtered_results["metadatas"][0]), start=1):
  print(f"Rank {rank} | Distance: {dist:.4f} | Subject: {meta['subject']}")
  print(f"  Document: {doc}")
  print()
  print("Notice: Only ML documents appear, even though ETL and Pandas might be 'related'.")

FILTERED QUERY: 'How do computers learn from examples?'
Filter : Only Machine Learning documents
Rank 1 | Distance: 1.0226 | Subject: Machine Learning
  Document: machine learning models learn patterns from training data

Notice: Only ML documents appear, even though ETL and Pandas might be 'related'.


In [ ]:
print("DISTANCE TO SIMILARITY CONVERSION")
print("="*50)
print(f"{'Distance':<15} {'Similarity':<15}{'Interpretation':<20}")
print("-"*50)

distance = [0.05, 0.20, 0.40, 0.65, 0.90]
interpretations = ["Near identical", "Very similar", "Related", "Somewhat related", "Not Related"]

for dist, interp in zip(distance, interpretations):
  similarity = 1 -dist
  print(f"{dist:<15.4f} {similarity:<15.4f} {interp:<20}")


DISTANCE TO SIMILARITY CONVERSION
Distance        Similarity     Interpretation      
--------------------------------------------------
0.0500          0.9500          Near identical      
0.2000          0.8000          Very similar        
0.4000          0.6000          Related             
0.6500          0.3500          Somewhat related    
0.9000          0.1000          Not Related         


In [ ]:
notes_df = pd.read_csv('college_notes.csv')
print("Dataset loaded!")
print(f"Shape of dataset: {notes_df.shape}")
print(f"Columns in dataset: {notes_df.columns.tolist()}")


Dataset loaded!
Shape of dataset: (15, 4)
Columns in dataset: ['note_id', 'subject', 'topic', 'content']


In [ ]:
print(f"First 5 rows of dataset:")
notes_df.head()

First 5 rows of dataset:


,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
print("Notes per subject:")
print(notes_df['subject'].value_counts())

Notes per subject:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64


In [ ]:
first_note = notes_df.iloc[0]
print(f"Note ID: {first_note['note_id']}")
print(f"Subject: {first_note['subject']}")
print(f"Topic: {first_note['content']}")
print(f"Content: {first_note['topic']}")


Note ID: N001
Subject: Data Engineering
Topic: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
Content: ETL Pipelines


In [ ]:
all_documents = notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()
all_metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for subject, topic in zip(notes_df['subject'], notes_df['topic'])
]

#MINI PROJECT  -- Smart Notes Search Engine
Now we combine everything to build a working semantic search system.

**Project Overview**


**Goal** : build a search engine that finds relevant college notes by MEANING , not just keywords

**Steps**:
1. create a new chromaDB collection for college notes
2. index all 15 notes in the collection
3. run semantic queries and display top result
4. filter results by subjects
5.

In [ ]:
!pip install chromadb sentence-transformers pandas -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv("college_notes.csv")

df.head()

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
print(df.shape)
print(df.columns)

(15, 4)
Index(['note_id', 'subject', 'topic', 'content'], dtype='object')


In [ ]:
#create chromodb collection
client = chromadb.Client()

collection = client.create_collection(
    name="college_notes"
)

print("Collection Created")

Collection Created


In [ ]:
#index all notes
collection.add(
    ids=df["note_id"].astype(str).tolist(),
    documents=df["content"].tolist(),
    metadatas=[
        {
            "subject": row["subject"],
            "topic": row["topic"]
        }
        for _, row in df.iterrows()
    ]
)

print("All notes indexed successfully")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 66.5MiB/s]


All notes indexed successfully


In [ ]:
print("Total Notes:", collection.count())

Total Notes: 15


In [ ]:
#sementic search
query = "How do we collect data from websites and applications?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, doc in enumerate(results["documents"][0],1):
    print(f"{i}. {doc}")
    print()

1. An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

2. A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.

3. Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.



In [ ]:
#Display Topic and Subject
query = "How do we collect data from websites and applications?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):

    print("Subject:",
          results["metadatas"][0][i]["subject"])

    print("Topic:",
          results["metadatas"][0][i]["topic"])

    print("Content:")
    print(results["documents"][0][i])

    print("-"*60)

Subject: Data Engineering
Topic: APIs and Data Collection
Content:
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
------------------------------------------------------------
Subject: Data Engineering
Topic: SQL Databases
Content:
A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
------------------------------------------------------------
Subject: Python Programming
Topic: Pandas Library
Content:
Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.
--------

In [ ]:
query = "How can machine learning understand language?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for doc in results["documents"][0]:
    print(doc)
    print()

A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-like text answer questions summarize documents and perform many language tasks. Examples include GPT Claude and LLaMA.

Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.

Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.



In [ ]:
#filter by subject
query = "How do chatbots generate responses?"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"subject":"Artificial Intelligence"}
)

for doc in results["documents"][0]:
    print(doc)
    print()

In [ ]:
#keyword search function
def keyword_search(query):

    query = query.lower()

    matches = []

    for _, row in df.iterrows():

        if query in row["content"].lower():

            matches.append({
                "topic": row["topic"],
                "content": row["content"]
            })

    return matches

In [ ]:
#Compare Keyword vs Semantic Search
query = "systems that learn patterns from data"

In [ ]:
print("KEYWORD SEARCH")
print("="*50)

keyword_results = keyword_search(query)

print(keyword_results)

KEYWORD SEARCH
[]


In [ ]:
print("KEYWORD SEARCH")
print("="*50)

keyword_results = keyword_search(query)

print(keyword_results)

KEYWORD SEARCH
[]


In [ ]:
print("SEMANTIC SEARCH")
print("="*50)

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):

    print("Topic:",
          results["metadatas"][0][i]["topic"])

    print()

SEMANTIC SEARCH
Topic: Supervised Learning

Topic: Decision Trees

Topic: Data Visualization



In [ ]:
#create results dataframe
query = "database management"

results = collection.query(
    query_texts=[query],
    n_results=5
)

output = pd.DataFrame({
    "Topic":[m["topic"] for m in results["metadatas"][0]],
    "Subject":[m["subject"] for m in results["metadatas"][0]],
    "Content":results["documents"][0]
})

output

,Topic,Subject,Content
0,SQL Databases,Data Engineering,A database is an organized collection of data ...
1,Data Cleaning,Data Engineering,Data cleaning involves fixing or removing inco...
2,Supervised Learning,Machine Learning,Supervised learning is a type of machine learn...
3,Decision Trees,Machine Learning,A decision tree is a machine learning model th...
4,Large Language Models,Generative AI,A Large Language Model or LLM is an AI model t...


#PRACTICE QUESTIONS -- DAY 7

answer these in the cells below. they cover all three units
1. Q1(Conceptual) :What is the difference between keywords search and semantic search? give one real-world example for each

2. Q2(Technical) : What does ```model.encode(['hello world'])``` return?What is the shape of the output?

3. Q3(critical thinking): A student adds document using model A but queries using model B. will results be correct? why or why not?

4. Q4(Application) : ChromaDB retuns distance [0.12, 0.45, 0.87].Which is most relevant? Which is the least relevant?

5. Q5(Code) : write the ```collection.add()``` call to store

### Q1 (Conceptual): What is the difference between keyword search and semantic search? Give one real-world example for each.

**Keyword Search**: Matches documents based on the presence of exact words or phrases in the query. It's good for precise, literal matches.
*   **Example**: Searching for "red shoes" on an e-commerce site will only show products with the exact phrase "red shoes" in their description.

**Semantic Search**: Understands the meaning and context of the query, returning results that are conceptually related, even if they don't contain the exact keywords. It uses embeddings to capture meaning.
*   **Example**: Searching for "comfortable footwear for running" might return results for "running sneakers" or "athletic shoes" even if the word "footwear" isn't present.

### Q2 (Technical): What does `model.encode(['hello world'])` return? What is the shape of the output?

`model.encode(['hello world'])` returns a NumPy array containing the vector embedding (a list of numbers) for the input sentence "hello world".

**Shape of the output**: `(1, 384)`
*   `1` represents the number of sentences encoded (in this case, one sentence).
*   `384` represents the dimensionality of the embedding vector produced by the `all-MiniLM-L6-v2` model.

### Q3 (Critical Thinking): A student adds documents using model A but queries using model B. Will results be correct? Why or why not?

No, the results will likely **not be correct** or at least suboptimal. Here's why:

*   **Incompatible Vector Spaces**: Each embedding model (Model A and Model B) creates its own unique vector space where it places sentences based on its learned understanding of meaning. If documents are embedded using Model A, they reside in Model A's vector space. If queries are embedded using Model B, they are in Model B's vector space.
*   **Meaningful Comparison Fails**: Comparing a vector from Model A's space with a vector from Model B's space using distance metrics (like cosine similarity) is like comparing apples and oranges. The distances or similarities calculated will not accurately reflect the semantic relationship between the query and the documents because their numerical representations are not aligned.

For accurate semantic search, the same embedding model **must be used** for both indexing (adding documents) and querying.

### Q4 (Application): ChromaDB returns distances [0.12, 0.45, 0.87]. Which is most relevant? Which is the least relevant?

In ChromaDB (and many other vector databases), a **lower distance indicates higher relevance**.

*   **Most relevant**: **0.12** (This is the smallest distance, indicating the closest match).
*   **Least relevant**: **0.87** (This is the largest distance, indicating the furthest match).

### Q5 (Code): Write the `collection.add()` call to store your `df` DataFrame into a ChromaDB collection named `my_notes`, with `note_id` as IDs, `content` as documents, and both `subject` and `topic` as metadata.

In [ ]:
# Assuming 'df' is already loaded and contains 'note_id', 'subject', 'topic', and 'content' columns

import chromadb

# Create a new client (or use an existing one)
client_q5 = chromadb.Client() # For in-memory, or chromadb.PersistentClient(path='./my_db') for persistent

# Get or create the collection
collection_q5 = client_q5.get_or_create_collection(name="my_notes")

# Add documents to the collection
collection_q5.add(
    ids=df["note_id"].astype(str).tolist(), # IDs must be strings
    documents=df["content"].tolist(),
    metadatas=[
        {"subject": row["subject"], "topic": row["topic"]}
        for index, row in df.iterrows()
    ]
)

print(f"Added {collection_q5.count()} documents to 'my_notes' collection.")

Added 15 documents to 'my_notes' collection.
